In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('../..'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


In [ ]:
import duckdb

DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
pm_df = con.execute("SELECT * FROM opspatial WHERE total_reception_events >= 15 AND total_release_events >= 15 AND position != 'Goalkeeper'").df()
con.close()

In [ ]:
pm_df["avg_reception_simpnorm_y"] = np.where(
    pm_df["average_reception_y"] <= 40,
    pm_df["average_reception_y"],
    80 - pm_df["average_reception_y"],
)

pm_df["avg_release_simpnorm_y"] = np.where(
    pm_df["average_release_y"] <= 40,
    pm_df["average_release_y"],
    80 - pm_df["average_release_y"],
)

pm_df["avg_reception_simpnorm_y_abs"] = pm_df["avg_reception_simpnorm_y"].abs()
pm_df["avg_release_simpnorm_y_abs"] = pm_df["avg_release_simpnorm_y"].abs()

feature_cols = [
    "average_reception_x",
    "avg_reception_simpnorm_y_abs",
    "average_release_x",
    "avg_release_simpnorm_y_abs",
]
pm_X = pm_df[feature_cols].dropna()
pm_X_scaled = StandardScaler().fit_transform(pm_X)

pm_k_range = range(2, 11)
pm_inertia = []
pm_silhouette_scores = []

for n_clusters in pm_k_range:
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = model.fit_predict(pm_X_scaled)

    pm_inertia.append(model.inertia_)
    pm_silhouette_scores.append(
        silhouette_score(
            pm_X_scaled,
            labels,
            sample_size=min(5000, len(pm_X_scaled)),
            random_state=42,
        )
    )

fig_pm, (ax1_pm, ax2_pm) = plt.subplots(1, 2, figsize=(14, 5))

ax1_pm.plot(list(pm_k_range), pm_inertia, marker="o")
ax1_pm.set(title="Elbow Method", xlabel="Number of Clusters (k)", ylabel="Inertia")

ax2_pm.plot(list(pm_k_range), pm_silhouette_scores, marker="o")
ax2_pm.set(title="Silhouette Scores", xlabel="Number of Clusters (k)", ylabel="Silhouette Score")

plt.tight_layout()
plt.show()

In [ ]:
from mplsoccer import Pitch
from IPython.display import display
from matplotlib.lines import Line2D

pm_X = pm_df[feature_cols].dropna()
pm_X_scaled = StandardScaler().fit_transform(pm_X)
cluster_data = pm_df.loc[pm_X.index].copy()
cluster_counts = []

for n_clusters in [3, 4, 5, 6]:
    model = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    cluster_data[f"cluster_k{n_clusters}"] = model.fit_predict(pm_X_scaled)
    pm_df[f"cluster_k{n_clusters}"] = cluster_data[f"cluster_k{n_clusters}"]

    counts = (
        cluster_data.groupby(f"cluster_k{n_clusters}")
        .size()
        .rename("player_match_pair_rows")
        .reset_index()
        .rename(columns={f"cluster_k{n_clusters}": "cluster"})
    )
    counts.insert(0, "k", n_clusters)
    cluster_counts.append(counts)

    pitch = Pitch(pitch_type="statsbomb", pitch_color="white", line_color="#333333")
    fig, ax = pitch.draw(figsize=(11, 7))
    color_map = plt.get_cmap("tab10", n_clusters)
    cluster_col = f"cluster_k{n_clusters}"

    for cluster in range(n_clusters):
        cluster_rows = cluster_data[cluster_data[cluster_col] == cluster]
        color = color_map(cluster)
        pitch.lines(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            alpha=0.25,
            lw=0.7,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_reception_x"],
            cluster_rows["average_reception_y"],
            color=color,
            marker="o",
            s=24,
            ax=ax,
        )
        pitch.scatter(
            cluster_rows["average_release_x"],
            cluster_rows["average_release_y"],
            color=color,
            marker="^",
            s=28,
            ax=ax,
        )

    cluster_handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=color_map(cluster), label=f"Cluster {cluster}")
        for cluster in range(n_clusters)
    ]
    cluster_handles.extend(
        [
            Line2D([0], [0], marker="o", color="#333333", linestyle="none", label="Reception"),
            Line2D([0], [0], marker="^", color="#333333", linestyle="none", label="Release"),
        ]
    )
    ax.legend(handles=cluster_handles, loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.set_title(f"Player-match spatial clusters (k={n_clusters})")
    plt.tight_layout()
    plt.show()

cluster_count_df = pd.concat(cluster_counts, ignore_index=True)
display(cluster_count_df)

In [ ]:
cluster_k3_0_df = pm_df[pm_df["cluster_k3"] == 0].copy()
cluster_k3_1_df = pm_df[pm_df["cluster_k3"] == 1].copy()
cluster_k3_2_df = pm_df[pm_df["cluster_k3"] == 2].copy()

cluster_k4_0_df = pm_df[pm_df["cluster_k4"] == 0].copy()
cluster_k4_1_df = pm_df[pm_df["cluster_k4"] == 1].copy()
cluster_k4_2_df = pm_df[pm_df["cluster_k4"] == 2].copy()
cluster_k4_3_df = pm_df[pm_df["cluster_k4"] == 3].copy()

cluster_k5_0_df = pm_df[pm_df["cluster_k5"] == 0].copy()
cluster_k5_1_df = pm_df[pm_df["cluster_k5"] == 1].copy()
cluster_k5_2_df = pm_df[pm_df["cluster_k5"] == 2].copy()
cluster_k5_3_df = pm_df[pm_df["cluster_k5"] == 3].copy()
cluster_k5_4_df = pm_df[pm_df["cluster_k5"] == 4].copy()

cluster_k6_0_df = pm_df[pm_df["cluster_k6"] == 0].copy()
cluster_k6_1_df = pm_df[pm_df["cluster_k6"] == 1].copy()
cluster_k6_2_df = pm_df[pm_df["cluster_k6"] == 2].copy()
cluster_k6_3_df = pm_df[pm_df["cluster_k6"] == 3].copy()
cluster_k6_4_df = pm_df[pm_df["cluster_k6"] == 4].copy()
cluster_k6_5_df = pm_df[pm_df["cluster_k6"] == 5].copy()

In [ ]:
positions_k3_0 = cluster_k3_0_df["position"].unique()
positions_k3_1 = cluster_k3_1_df["position"].unique()
positions_k3_2 = cluster_k3_2_df["position"].unique()

positions_k4_0 = cluster_k4_0_df["position"].unique()
positions_k4_1 = cluster_k4_1_df["position"].unique()
positions_k4_2 = cluster_k4_2_df["position"].unique()
positions_k4_3 = cluster_k4_3_df["position"].unique()

positions_k5_0 = cluster_k5_0_df["position"].unique()
positions_k5_1 = cluster_k5_1_df["position"].unique()
positions_k5_2 = cluster_k5_2_df["position"].unique()
positions_k5_3 = cluster_k5_3_df["position"].unique()
positions_k5_4 = cluster_k5_4_df["position"].unique()

positions_k6_0 = cluster_k6_0_df["position"].unique()
positions_k6_1 = cluster_k6_1_df["position"].unique()
positions_k6_2 = cluster_k6_2_df["position"].unique()
positions_k6_3 = cluster_k6_3_df["position"].unique()
positions_k6_4 = cluster_k6_4_df["position"].unique()
positions_k6_5 = cluster_k6_5_df["position"].unique()

In [ ]:
position_counts_k3_0 = cluster_k3_0_df["position"].value_counts()
position_counts_k3_1 = cluster_k3_1_df["position"].value_counts()
position_counts_k3_2 = cluster_k3_2_df["position"].value_counts()

position_counts_k4_0 = cluster_k4_0_df["position"].value_counts()
position_counts_k4_1 = cluster_k4_1_df["position"].value_counts()
position_counts_k4_2 = cluster_k4_2_df["position"].value_counts()
position_counts_k4_3 = cluster_k4_3_df["position"].value_counts()

position_counts_k5_0 = cluster_k5_0_df["position"].value_counts()
position_counts_k5_1 = cluster_k5_1_df["position"].value_counts()
position_counts_k5_2 = cluster_k5_2_df["position"].value_counts()
position_counts_k5_3 = cluster_k5_3_df["position"].value_counts()
position_counts_k5_4 = cluster_k5_4_df["position"].value_counts()

position_counts_k6_0 = cluster_k6_0_df["position"].value_counts()
position_counts_k6_1 = cluster_k6_1_df["position"].value_counts()
position_counts_k6_2 = cluster_k6_2_df["position"].value_counts()
position_counts_k6_3 = cluster_k6_3_df["position"].value_counts()
position_counts_k6_4 = cluster_k6_4_df["position"].value_counts()
position_counts_k6_5 = cluster_k6_5_df["position"].value_counts()

In [ ]:
position_counts_df = pd.concat(
    {
        "cluster_k3_0": position_counts_k3_0,
        "cluster_k3_1": position_counts_k3_1,
        "cluster_k3_2": position_counts_k3_2,
        "cluster_k4_0": position_counts_k4_0,
        "cluster_k4_1": position_counts_k4_1,
        "cluster_k4_2": position_counts_k4_2,
        "cluster_k4_3": position_counts_k4_3,
        "cluster_k5_0": position_counts_k5_0,
        "cluster_k5_1": position_counts_k5_1,
        "cluster_k5_2": position_counts_k5_2,
        "cluster_k5_3": position_counts_k5_3,
        "cluster_k5_4": position_counts_k5_4,
        "cluster_k6_0": position_counts_k6_0,
        "cluster_k6_1": position_counts_k6_1,
        "cluster_k6_2": position_counts_k6_2,
        "cluster_k6_3": position_counts_k6_3,
        "cluster_k6_4": position_counts_k6_4,
        "cluster_k6_5": position_counts_k6_5,
    },
    names=["cluster", "position"],
).rename("count").reset_index()

display(position_counts_df)

In [ ]:
position_cluster_counts_df = (
    position_counts_df.pivot(
        index="position",
        columns="cluster",
        values="count",
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)
position_cluster_counts_df.columns.name = None

display(position_cluster_counts_df)